# LoRA SFT on Kaggle

Run the cells top to bottom. **In notebook cells, shell commands need the `!` prefix.**
In the Kaggle *terminal* you run the same commands without `!`.

Before starting, in the right-hand panel:
1. **Session options -> Accelerator -> GPU T4 x2** (or P100)
2. **Session options -> Internet -> On** (needed to download the model from Hugging Face)
3. **Input -> Upload -> Dataset**, and upload `lahja-colab.tgz`, built on your Mac with:
```bash
COPYFILE_DISABLE=1 tar czf ~/Desktop/lahja-colab.tgz --exclude '__pycache__' --exclude '._*' \
    src pyproject.toml README.md configs data/sft \
    data/processed/egy_test.jsonl data/processed/massive_ar_test.jsonl
```
Use `tar`, not Finder's "Compress": a Finder zip adds AppleDouble files (`._sft_lora.py`) that
can shadow the real modules.

In [ ]:
# 1. Check the GPU. T4/P100 have no bfloat16 - the training script falls back to fp16 by itself.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Find the uploaded bundle (it appears under /kaggle/input/<your-dataset-name>/)
!ls -R /kaggle/input | head -20

In [ ]:
# 3. Extract into the writable working dir. Fix the path if your dataset is named differently.
BUNDLE = "/kaggle/input/lahja-colab/lahja-colab.tgz"

!mkdir -p /kaggle/working/lahja && tar xzf {BUNDLE} -C /kaggle/working/lahja
%cd /kaggle/working/lahja
# Strip macOS AppleDouble junk if the archive carried any.
!find /kaggle/working/lahja -name "._*" -delete
# The real module files must be listed here (no "._" prefixes):
!ls src/lahja/train/ && ls data/sft/*

In [ ]:
# 4. Install. `%pip` (not `!pip`) installs into THIS kernel's environment - on Kaggle `!pip`
# and `!python` can resolve to a different interpreter, which gives "No module named 'lahja'".
%pip install -q -e .
%pip install -q -U "trl>=1.13" peft datasets accelerate
# Kaggle/Colab ship an old torchao that current peft rejects; we never use it.
%pip uninstall -q -y torchao
import torch

print(torch.__version__, torch.cuda.get_device_name(0), "bf16:", torch.cuda.is_bf16_supported())

## Train

Defaults match the Mac track: 1,200 steps x batch 16, LoRA rank 16, loss on the assistant JSON
only. **Watch the first 100 steps**: loss should fall below ~0.5 with no jump above 1.0.
If it spikes, stop and rerun with `--lr 1e-4`.

`CUDA_VISIBLE_DEVICES=0` pins training to one GPU - simpler and faster than splitting a small
model across the two T4s.

In [ ]:
import os
import sys

ROOT = "/kaggle/working/lahja"  # where cell 3 extracted the bundle
os.chdir(ROOT)  # must run from the project root, NOT from /kaggle/working
# Point at the real package. Without this, a folder named "lahja" in the parent directory
# shadows it as a namespace package: `import lahja` works but `lahja.train` is missing.
os.environ["PYTHONPATH"] = f"{ROOT}/src"
# $PY is this kernel's own interpreter - the one %pip installed into.
os.environ["PY"] = sys.executable
os.environ["MODEL"] = "Qwen/Qwen3-1.7B"
os.environ["TAG"] = "qwen3-17b"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("cwd:", os.getcwd())
!$PY -c "import lahja.train.sft_lora as m; print('resolved:', m.__file__)"

In [ ]:
# C = Saudi/MSA only. This is the run that completes the 2x2 comparison.
!$PY -m lahja.train.sft_lora --model $MODEL --mix C --out adapters/C-$TAG

In [ ]:
# D = Saudi/MSA + Egyptian synthetic (skip if you already have D from Colab)
!$PY -m lahja.train.sft_lora --model $MODEL --mix D --out adapters/D-$TAG

In [ ]:
# 5. Evaluate with the same code used for every other model in the table
!$PY -m lahja.eval.run_eval --backend hf --model $MODEL --adapter adapters/C-$TAG --prompt sft --run-name C-$TAG --eval-set data/processed/egy_test.jsonl --eval-set data/processed/massive_ar_test.jsonl --limit 300

In [ ]:
# 6. Package results + adapters. Download them from the Output panel on the right.
!tar czf /kaggle/working/lahja-results.tgz results adapters
!ls -lh /kaggle/working/lahja-results.tgz

On the Mac, from the project root: `tar xzf ~/Downloads/lahja-results.tgz`

Then `make report` to fold the new rows into `results/summary.md`.

**Kaggle sessions end after ~12 h (or ~20 min idle)** and `/kaggle/working` is wiped, so
download the archive as soon as training finishes.